# Joint dual run: input construction and prediction review

This is a **saved-artifact** notebook. It does not connect to HPC, reload a large dataset, train, or execute a model. Select a downloaded dual run, then inspect its input audit, loss histories, UT/FT curves and displacement fields together.

Without a run path, the notebook opens the local input-construction illustrations only. Those are not prediction results.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import resources
from resources.MLmetrics import plot_dual_input_example, plot_dual_predictions, plot_loss_history

REPO = Path(resources.__file__).resolve().parents[1]
RUN_DIR = REPO / 'data/MULTI/Dual/Transformer/dual-MULTI-test-260907'
SAVE_FIGURES = True

results = RUN_DIR / 'results'
if not (results / 'input_audit/input_examples.npz').is_file():
    raise FileNotFoundError('Download the run with B3_ML-transfer-mac.sh, or select another local RUN_DIR.')
metadata = json.loads((RUN_DIR / 'model_data.json').read_text())
spec = metadata['metadata'].get('context_spec', {})
audit_dir = results / 'input_audit'
display(Markdown(f'**Run:** {RUN_DIR.name}. This three-epoch smoke run tests execution, not predictive quality.'))

with np.load(audit_dir / 'input_examples.npz', allow_pickle=False) as file:
    audit = {key: file[key] for key in file.files}

**ILLUSTRATIONS ONLY — no saved model performance is being displayed.**

## 1. Follow one input through feature construction

The run saves three training-input examples by default: first, middle and last in split order, not a statistical sample of extremes. Change `INPUT_EXAMPLE` to compare them. The accompanying Markdown includes explicit node-by-node arithmetic. x₀/y₀/designable are shared; UT interfaces and FT coupling/tip features are task-specific. They enter **before** attention. A coupling flag is not a hard zero-displacement condition.

In [ ]:
INPUT_EXAMPLE = 0
fig = plot_dual_input_example(audit['canonical_coords'], audit['raw_disorder'][INPUT_EXAMPLE],
                             {m: audit[f'{m}_context'][INPUT_EXAMPLE] for m in ('UT','FT')},
                             {m: audit[f'{m}_node_mask'] for m in ('UT','FT')},
                             audit['feature_names'].tolist(), spec, audit['sample_ids'][INPUT_EXAMPLE])
plt.show()
report_path = audit_dir / f'sample-{INPUT_EXAMPLE+1:02d}.md'
# Plot already shown above; remove only the duplicate image line from displayed Markdown.
report = '\n'.join(line for line in report_path.read_text().splitlines() if not line.startswith('!['))
display(Markdown(report))

## 2. What reached the shared encoder at a selected node?

This compact comparison displays the actual saved context channels (including sample-specific pin decisions), not just their reference defaults. Every row is a feature, not an extra input node. The disorder projection and context projection have the same latent width and are added to the task embedding.

In [ ]:
NODE_INDEX = 485  # Canonical index for (55,35) in illustrations; choose explicitly for a real run.
if not 0 <= NODE_INDEX < len(audit['canonical_coords']):
    raise IndexError('NODE_INDEX is outside this run.')
display(pd.DataFrame({m: audit[f'{m}_context'][INPUT_EXAMPLE, NODE_INDEX] for m in ('UT','FT')},
                     index=audit['feature_names']).rename_axis('Named feature'))
print('Reference position:', audit['canonical_coords'][NODE_INDEX])
print('Raw disorder:', audit['raw_disorder'][INPUT_EXAMPLE, NODE_INDEX])
print('Standardised disorder:', audit['normalized_disorder'][INPUT_EXAMPLE, NODE_INDEX])

## 3. Training history: joint objective and its four contributions

Weighted contributions add to the joint objective; they need not have equal magnitudes. These are training-space losses, not physical RMSE. A descending total alone does not show that every task is improving.

In [ ]:
if results is not None:
    history = pd.read_csv(RUN_DIR / 'loss_history.csv')
    plot_loss_history(history)
    plt.show()
    fig, axes = plt.subplots(2,2,figsize=(11,7),constrained_layout=True)
    for ax,(kind,mode) in zip(axes.flat, [('field','UT'),('field','FT'),('curve','UT'),('curve','FT')]):
        for split in ('train','val'):
            ax.plot(history['epoch'],history[f'{split}_weighted_{kind}_{mode}'],label=split)
        ax.set(title=f'{mode} {kind}: weighted contribution',xlabel='Epoch',ylabel='Loss contribution')
        ax.legend()
    plt.show()
    if SAVE_FIGURES:
        output = results / 'postProcessing'
        output.mkdir(exist_ok=True)
        fig.savefig(output / 'four-loss-terms.png',dpi=160)
else:
    print('No run selected; no training history to show.')

## 4. Paired curves and field maps for the same sample

Choose sample index, frame index and component beside the plot. The same frame index can correspond to different UT/FT load fractions; each row labels its actual frame value. Invalid values and absent FT nodes are excluded. Truth and prediction share one colour scale **within each task**, while the error plot has its own scale. Fields are plotted at reference positions; they are not displaced-mesh plots.

In [ ]:
PREDICTION_SAMPLE = 0
FRAME_INDEX = -1
COMPONENT_INDEX = 1  # U2 when components are [U1,U2]; select 0 for a one-component run.
if results is not None:
    metrics = json.loads((results / 'metrics.json').read_text())
    split = metrics['evaluation_split']
    with np.load(results / 'predictions.npz',allow_pickle=False) as file:
        arrays = {key: file[key] for key in file.files}
    curve_fig, field_fig = plot_dual_predictions(arrays, split, PREDICTION_SAMPLE, FRAME_INDEX, COMPONENT_INDEX)
    plt.show()
    display(Markdown(f"Evaluation split: **{split}**. These are end-to-end predictions from disorder, not true-field curve oracles."))
    for kind in ('field','curve'):
        for mode in ('UT','FT'):
            summary = metrics['diagnostics'][kind][mode]
            display(Markdown(f"**{mode} {kind}:** RMSE = {summary['rmse']:.5g} in the saved physical units. Do not compare field/curve or UT/FT RMSE numerically as though they share units."))
    if SAVE_FIGURES:
        output = results / 'postProcessing'
        output.mkdir(exist_ok=True)
        curve_fig.savefig(output / 'paired-curves.png',dpi=160)
        field_fig.savefig(output / 'paired-fields.png',dpi=160)
else:
    print('No real run selected. No response curves or displacement fields are invented for the illustrations.')